# nnU-Net v2 ResEnc semantic segmentation with dataset-fixer

Dataset conversion, standardized W&B config/tags, portable bundle manifests, ZIP creation,
and W&B upload are handled by dataset-fixer. The ZIP is always created locally and is never
copied to Google Drive. The final cell prints whether W&B accepted the upload, the run URL,
the remote file URL when available, and the exact local path/hash retained in every case.

In [ ]:
import importlib.util
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    !pip install --upgrade --upgrade-strategy only-if-needed "dataset-fixer @ git+https://github.com/mooch443/dataset-fixer.git"

## Configuration

### Common settings

Change these for a new dataset, geometry, fold, or training run.

In [ ]:
from datetime import datetime
from pathlib import Path
import torch

DATASET_SOURCE = (
    "/content/drive/MyDrive/islands/islands-fair-base-sem.zip"
    if IN_COLAB else
    "/Users/tristan/Downloads/island-dataset/islands-fair-base-sem"
)
NATIVE_TILE_SIZE = 128
UPSCALE_FACTOR = 2
EPOCHS = 100
FOLD = 0
USE_WANDB = True

### Advanced settings

These backend, execution, and naming defaults usually do not need editing.

In [ ]:
RESULTS_DIRECTORY = Path("/content/training-results" if IN_COLAB else "./training-results").resolve()
WORKERS = 4
MODEL_INPUT_SIZE = NATIVE_TILE_SIZE * UPSCALE_FACTOR
BASE_MODEL_NAME = "nnunet-resenc-m"
TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M")
MODEL_NAME = f"{BASE_MODEL_NAME}-{MODEL_INPUT_SIZE}px-{UPSCALE_FACTOR}x-{TIMESTAMP}"
PLANNER = "nnUNetPlannerResEncM"
CONFIGURATION = "2d"
TRAINER = f"nnUNetTrainer_{EPOCHS}epochs"
DEVICE_OVERRIDE = None  # None selects CUDA, then MPS, then CPU.
EVALUATION_BATCH_SIZE = -1
WANDB_ENTITY = "max-planck-institute-for-animal-behavior"
WANDB_PROJECT = "schools-segmentation"
WANDB_RUN_NAME = MODEL_NAME
WANDB_RUN_ID = None  # Optional stable ID; one is generated and saved beside the checkpoint otherwise.

device = DEVICE_OVERRIDE or (
    "cuda" if torch.cuda.is_available() else
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu"
)

In [ ]:
if USE_WANDB:
    !wandb login

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

## Preparation

In [ ]:
from dataset_fixer import Dataset
from dataset_fixer.convert import Kind, prepare

# Dataset.open accepts either a folder or ZIP and reuses a verified local extraction.
# NNUNET preparation builds the raw layout, deterministic split, dataset.json, plans,
# and preprocessed data. The content-addressed result is reused when the source,
# geometry, planner, and preprocessing settings are unchanged. Run-specific arguments
# populate prepared.config without making epochs or folds part of the dataset cache.
source_dataset = Dataset.open(DATASET_SOURCE)
prepared = prepare(
    source_dataset,
    Kind.NNUNET,
    name=MODEL_NAME,
    native_tile_size=NATIVE_TILE_SIZE,
    upscale_factor=UPSCALE_FACTOR,
    workers=WORKERS,
    preprocess=True,
    planner=PLANNER,
    trainer=TRAINER,
    configuration=CONFIGURATION,
    fold=FOLD,
    epochs=EPOCHS,
    device=device,
)
bundle_config = prepared.config
prepared

### Resolved backend configuration

In [ ]:
# The plans identifier and expected checkpoint are discovered/defaulted by prepare().
PLANS = str(bundle_config.model["plans"])
CHECKPOINT_NAME = str(bundle_config.model["checkpoint"])
print(f"Selected trainer: {TRAINER} ({EPOCHS} epochs)")
print(f"Selected ResEnc plan: {PLANNER} -> {PLANS}/{CONFIGURATION}")

## Train

In [ ]:
import os
import subprocess
import wandb
from dataset_fixer.wandb import configure as configure_wandb
from nnunetv2.utilities.file_path_utilities import get_output_folder

os.environ.update(prepared.backend["environment"])
DATASET_ID = int(prepared.backend["dataset_id"])
FOLD_OUTPUT = Path(get_output_folder(
    DATASET_ID, trainer_name=TRAINER, plans_identifier=PLANS,
    configuration=CONFIGURATION, fold=FOLD,
))
FOLD_OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_FOLDER = FOLD_OUTPUT.parent
RUN_ID_FILE = FOLD_OUTPUT / "wandb_run_id.txt"
WANDB_RUN_PATH = None
if USE_WANDB:
    if RUN_ID_FILE.is_file():
        wandb_run_id = RUN_ID_FILE.read_text(encoding="utf-8").strip()
        if WANDB_RUN_ID not in {None, wandb_run_id}:
            raise ValueError("Configured WANDB_RUN_ID conflicts with the ID saved for this fold")
    else:
        wandb_run_id = WANDB_RUN_ID or wandb.util.generate_id()
        RUN_ID_FILE.write_text(wandb_run_id, encoding="utf-8")
    os.environ.update({
        "nnUNet_wandb_enabled": "1",
        "nnUNet_wandb_project": WANDB_PROJECT,
        "nnUNet_wandb_mode": "online",
        "WANDB_ENTITY": WANDB_ENTITY,
        "WANDB_PROJECT": WANDB_PROJECT,
        "WANDB_NAME": WANDB_RUN_NAME,
        "WANDB_RUN_ID": wandb_run_id,
    })
    WANDB_RUN_PATH = f"{WANDB_ENTITY}/{WANDB_PROJECT}/{wandb_run_id}"
    print(f"nnU-Net W&B run: https://wandb.ai/{WANDB_RUN_PATH}")
else:
    os.environ["nnUNet_wandb_enabled"] = "0"

command = [
    "nnUNetv2_train", str(DATASET_ID), CONFIGURATION, str(FOLD),
    "-tr", TRAINER, "-p", PLANS, "-device", device,
]
if (FOLD_OUTPUT / "checkpoint_latest.pth").is_file():
    command.append("--c")
    print("Resuming the existing fold checkpoint and W&B run.")
print("Training command:", " ".join(command))
subprocess.run(command, check=True)
CHECKPOINT = FOLD_OUTPUT / CHECKPOINT_NAME
if not CHECKPOINT.is_file():
    raise FileNotFoundError(f"Training completed without expected checkpoint: {CHECKPOINT}")

if WANDB_RUN_PATH is not None:
    try:
        training_run = wandb.Api().run(WANDB_RUN_PATH)
        configure_wandb(training_run, bundle_config)
        print(f"Standardized W&B config written to: https://wandb.ai/{WANDB_RUN_PATH}")
    except Exception as exc:
        print(f"Could not configure the W&B training run: {type(exc).__name__}: {exc}")

## Evaluate results

In [ ]:
from dataset_fixer import Model

models = Model.load_many({
    "nnunet-resenc": {
        "source": MODEL_FOLDER,
        "folds": (FOLD,),
        "checkpoint": CHECKPOINT.name,
    }
}).configure({
    "nnunet-resenc": {
        "task": "semantic",
        "native_tile_size": NATIVE_TILE_SIZE,
        "upscale_factor": UPSCALE_FACTOR,
        "device": device,
        "workers": WORKERS,
        "batch_size": EVALUATION_BATCH_SIZE,
        "inference": "sahi",
    }
})
comparison = models.compare(source_dataset, split="val", save_prediction_plots=True)

In [ ]:
import wandb
from dataset_fixer.bundle import Outcome, create
from dataset_fixer.wandb import configure as configure_wandb
from dataset_fixer.wandb import upload

outcome = Outcome(
    checkpoint=CHECKPOINT,
    metrics={"comparison_report": str(comparison.location)},
    files={"model": MODEL_FOLDER},
)
model_bundle = create(bundle_config, outcome)
upload_run = None
if WANDB_RUN_PATH is not None:
    try:
        upload_run = wandb.Api().run(WANDB_RUN_PATH)
        configure_wandb(upload_run, bundle_config)
        tracked_keys = ("model_name", "framework", "task", "native_tile_size",
                        "upscale_factor", "model_input_size", "trainer", "planner",
                        "plans", "configuration", "fold", "epochs")
        print("Standardized W&B config:", {key: upload_run.config.get(key) for key in tracked_keys})
    except Exception as exc:
        print(f"Could not reopen W&B run {WANDB_RUN_PATH}: {type(exc).__name__}: {exc}")
if upload_run is not None:
    model_bundle = upload(upload_run, model_bundle, outcome)
elif USE_WANDB:
    print("W&B upload skipped because the intended run could not be reopened.")
print(f"Local bundle: {model_bundle.path}")
print(f"Size: {model_bundle.size:,} bytes")
print(f"SHA-256: {model_bundle.sha256}")
if model_bundle.uploaded:
    print("W&B sync: confirmed; upload completed and run summary updated.")
    print(f"W&B run: https://wandb.ai/{WANDB_RUN_PATH}")
    print(f"W&B file: {model_bundle.path.name}")
    if model_bundle.remote_url:
        print(f"Remote file: {model_bundle.remote_url}")
else:
    print("W&B sync: not completed; use the local bundle above.")
    for warning in model_bundle.warnings:
        print(f"- {warning}")